# Explainability — Why does the model predict pneumonia?

High recall alone does not tell us whether the ResNet-18 (`train_optimized.py`) looks at the **lungs** or at **shortcuts** such as text markers, hospital labels or image borders. This notebook explains the model's predictions with two methods:

1. **Grad-CAM** — uses the gradients of the predicted class flowing into the last convolutional block (`layer4`) to highlight the regions that drove the decision.
2. **Occlusion sensitivity** — hides one 15×15 patch at a time and measures how much the pneumonia probability drops. It does not depend on gradients, so it cross-checks Grad-CAM.

All code lives in `src/explainability/gradcam.py`; `explain.py` runs the full analysis from the command line.

**Prerequisite:** the dataset extracted to `data/interim/chest_xray` (see `setup.ipynb`) and `pneumonia_resnet18_recall_optimized.pth` in the repo root.

In [ ]:
import torch
import matplotlib.pyplot as plt
from IPython.display import Image as ShowImage

from src.explainability.gradcam import GradCAM, load_resnet18, border_fraction, denormalize, occlusion_sensitivity
from src.dataset import DATA_DIR, test_transform

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
model = load_resnet18("pneumonia_resnet18_recall_optimized.pth", device)

## 1. Run Grad-CAM on the whole test set

This writes per-outcome figures to `reports/figures/explainability/` and a per-image table to `reports/explainability_summary.csv`.

In [ ]:
!python explain.py --split test --num-examples 4 --num-occlusion 2

## 2. Inspect each outcome group

- **True positives / true negatives:** does the heatmap sit inside the lung fields?
- **False negatives (missed pneumonia):** the most dangerous errors — what did the model look at instead?
- **False positives:** with `pos_weight=3` and threshold 0.4 the model leans towards pneumonia; check whether these are driven by real opacities or by artefacts.

In [ ]:
for group in ["true_positive", "true_negative", "false_positive", "false_negative"]:
    try:
        display(ShowImage(f"reports/figures/explainability/{group}.png", width=450))
    except FileNotFoundError:
        print(f"No {group} examples")

## 3. Grad-CAM vs. occlusion

If both methods highlight the same area, the explanation is more trustworthy. Red in the occlusion map means hiding that patch *lowered* the pneumonia probability, i.e. the patch supported a pneumonia prediction.

In [ ]:
for group in ["true_positive", "false_positive", "false_negative"]:
    try:
        display(ShowImage(f"reports/figures/explainability/occlusion_{group}.png", width=650))
    except FileNotFoundError:
        print(f"No {group} examples")

## 4. Shortcut check: how much attention falls on the image border?

`cam_border_fraction` is the share of the Grad-CAM heatmap lying in the outer 15% band of the image. Lungs are central; markers, labels and scanner edges are near the border.

For reference, a perfectly flat heatmap gives **≈ 0.51** (the band covers 51% of the image area). Values clearly below that mean attention is concentrated centrally; values near or above it suggest the model is using border artefacts.

In [ ]:
import pandas as pd

summary = pd.read_csv("reports/explainability_summary.csv")
display(summary.groupby("outcome")["cam_border_fraction"].describe())

summary.boxplot(column="cam_border_fraction", by="outcome", figsize=(8, 4))
plt.axhline(0.51, color="gray", linestyle="--", label="flat heatmap")
plt.suptitle("")
plt.title("Grad-CAM attention on the image border")
plt.ylabel("border fraction")
plt.legend()
plt.show()

In [ ]:
# Images where the model looks most at the border — inspect these for shortcuts
summary.sort_values("cam_border_fraction", ascending=False).head(10)

## 5. Explain a single X-ray

In [ ]:
import os
example = os.path.join(DATA_DIR, "test", "PNEUMONIA", sorted(os.listdir(os.path.join(DATA_DIR, "test", "PNEUMONIA")))[0])
!python explain.py --image "{example}"
display(ShowImage(f"reports/figures/explainability/single_{os.path.splitext(os.path.basename(example))[0]}.png", width=650))

## 6. Findings

_Fill in after running:_
- Where does the model look for correct pneumonia predictions?
- What do the false negatives have in common?
- Do Grad-CAM and occlusion agree?
- Is there evidence of shortcut learning (high border fraction)?